# Promotion strategies during equivalent events and media releases

The RAG layer should not only retrieve related games. It should say **when** to promote them and **how**, using the real-world event or entertainment release as the clock.

Rule: a catalog product is merchandised during the timeframe of its event equivalent.

- EA Sports FC / FIFA / Football Manager → football leagues and World Cups
- Madden NFL → NFL season
- NBA 2K → NBA playoffs
- TopSpin / AO Tennis → Roland-Garros and other tennis windows
- Street Fighter movie or EVO → fighting-game SKUs
- Marvel's Spider-Man 2 → Spider-Man: Brand New Day theatrical window

Each plan has three phases: **lead-in**, **event runtime**, **afterglow**.

The calendar spans **2026–2030** and includes global/regional physical events, digital showcases/sales, esports, local conferences/festivals, and cross-media releases in theatrical, television, streaming, animation, anime, audio, reality, and live-stage formats. Announced-but-unreleased products are included in the catalog merge and correlated into event windows (including dedicated release-window rows). Exact dates and year-wide planning windows remain separately labeled.

In [ ]:
from datetime import date
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.documents import build_retrieval_corpus, keyword_retrieve
from src.load_data import load_adaptations, load_catalog, load_events
from src.promote import (
    build_plans,
    plans_active_on,
    retrieve_promotions,
    write_promotion_csv,
)

In [ ]:
events = load_events()
adaptations = load_adaptations()
catalog = load_catalog(games_only=True, drop_placeholder_dates=True)
plans = build_plans(events, adaptations, catalog)
print(f"promotion plans: {len(plans)}")
print(f"distinct events with equivalent SKUs: {len({plan['event'] for plan in plans})}")

## Worked example: EA Sports FC × FIFA Women's World Cup

Football products are promoted through the tournament window, not as an always-on listing. Current-gen FC is the hero; points packs are attach; last year can discount.

In [ ]:
world_cup = [plan for plan in plans if "Women's World Cup" in plan["event"]]
print(f"plans tied to FIFA Women's World Cup: {len(world_cup)}\n")
for plan in world_cup:
    print(f"{plan['canonical_title']}  [{plan['role']}]  {plan['platform']}")
    print(f"  event runtime : {plan['event_start']} → {plan['event_end']}")
    print(f"  promo window  : {plan['promo_start']} → {plan['promo_end']}")
    print(f"  offer         : {plan['offer']}")
    print(f"  {plan['strategy_summary']}")
    for phase in plan["phases"]:
        print(f"  {phase['label']} {phase['start']} → {phase['end']}")
        for tactic in phase["tactics"]:
            print(f"     - {tactic}")
    print()

## Other sport and IP equivalents

Same clock logic for leagues, esports, and adaptations.

In [ ]:
seen = set()
print(f"{'event':40} {'product':42} {'promo window'}")
print("-" * 110)
for plan in plans:
    if plan["role"] != "game":
        continue
    key = (plan["event"], plan["canonical_title"])
    if key in seen:
        continue
    seen.add(key)
    print(
        f"{plan['event'][:40]:40} {plan['canonical_title'][:42]:42} "
        f"{plan['promo_start']} → {plan['promo_end']}"
    )

## What should we promote on a given day?

`plans_active_on` uses the full promotion window (lead-in through afterglow), not only the event dates.

In [ ]:
day = date(2027, 6, 15)
active = plans_active_on(plans, day)
print(f"active on {day}: {len(active)} plan rows\n")
for plan in active:
    if plan["role"] != "game":
        continue
    live = plan["event_start"] <= day.isoformat() <= plan["event_end"]
    phase = "LIVE" if live else "surround"
    print(f"  [{phase:9}] {plan['canonical_title'][:40]:40} ← {plan['event']}")

## RAG retrieve: marketing question → timed plan

Promotion plans are retrieval chunks. Asking how to market a franchise should return the equivalent event window and tactics — not a generic product card.

In [ ]:
corpus = build_retrieval_corpus(events, adaptations, [], plans)
print(f"corpus size: {len(corpus)}  (includes {sum(1 for d in corpus if d['kind']=='promotion')} promotion chunks)\n")

for query in [
    "How should we promote EA Sports FC during football tournaments?",
    "Madden NFL marketing during the NFL season",
    "Tennis game promotion around Roland-Garros",
    "Resident Evil promotion when the movie comes out",
]:
    print(f"Q: {query}")
    for score, plan in retrieve_promotions(plans, query, limit=3):
        print(
            f"  {score:.2f}  {plan['canonical_title'][:34]:34}  "
            f"{plan['event'][:28]:28}  {plan['promo_start']} → {plan['promo_end']}"
        )
    print()

In [ ]:
path = write_promotion_csv(plans)
print(f"wrote {path}")